# 00 Data splits

Loads the balanced dataset and splits it once into training, validation and test sets, **stratified by family**, so that every family (and `legit`) appears in all three sets in the same 80/10/10 proportion. The splits are saved to `data/splits/`, and every model and experiment reads these files, so we always train, tune and test on exactly the same rows.

Steps:
1. Load the balanced data
2. Split it into train / validation / test, stratified by family
3. Check the split
4. Save the splits and their fingerprints (`manifest.json`)
5. Reload the saved files and verify them

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BALANCED = ROOT / "data/processed/domains_balanced.csv.gz"
SPLITS = ROOT / "data/splits"
SEED = 42                 # fixed, so the split is the same on every machine
VAL_SIZE = TEST_SIZE = 0.10

## 1. Load the balanced data

Built by `src/data/build_dataset.py` and `src/data/balance_dataset.py`. Columns: `domain`, `label` (`legit` / `dga`) and `family` (`legit`, `other_dga` or one of 102 DGA families).

In [2]:
df = pd.read_csv(BALANCED, dtype=str, keep_default_na=False)
print(f"{len(df):,} domains, {df['family'].nunique()} classes "
      f"(legit + {df.loc[df['label'] == 'dga', 'family'].nunique()} DGA families)")
df.head()

1,999,335 domains, 104 classes (legit + 103 DGA families)


,domain,label,family
0,lkcoffee.com,legit,legit
1,auunonypqytl.ru,dga,cryptolocker
2,yhwbmwfqulkt.net,dga,tinba
3,ses4.com,legit,legit
4,vertextodaymarket.it,legit,legit


In [3]:
df["label"].value_counts()

label
dga      999671
legit    999664
Name: count, dtype: int64

In [4]:
family_sizes = df["family"].value_counts()
print("smallest families:")
family_sizes.tail(5)

smallest families:


family
sinowal                396
bigviktor_subdomain    394
sharkbot               350
vidro_subdomain        330
lechiket               300
Name: count, dtype: int64

## 2. Split, stratified by family

Stratifying by `family` (not only by `label`) keeps each family's share the same in every set, so each family has validation and test examples. Since `legit` is one of the families, the DGA/legitimate balance is kept too.

First 80% train vs. 20% held out, then the held-out part 50/50 into validation and test.

In [5]:
train, held_out = train_test_split(
    df, test_size=VAL_SIZE + TEST_SIZE, stratify=df["family"], random_state=SEED)
val, test = train_test_split(
    held_out, test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE),
    stratify=held_out["family"], random_state=SEED)

splits = {"train": train, "val": val, "test": test}
for name, part in splits.items():
    print(f"{name:5s} {len(part):>9,} rows ({len(part) / len(df):.1%})")

train 1,599,468 rows (80.0%)
val     199,933 rows (10.0%)
test    199,934 rows (10.0%)


## 3. Check the split

**Every family is in every set**, with about 80/10/10 of its domains.

In [6]:
per_family = pd.DataFrame({name: part["family"].value_counts()
                           for name, part in splits.items()}).fillna(0).astype(int)
per_family["total"] = per_family.sum(axis=1)
per_family = per_family.sort_values("total")

missing = per_family[(per_family[["train", "val", "test"]] == 0).any(axis=1)]
assert missing.empty, f"families missing from a split: {list(missing.index)}"
print(f"all {len(per_family)} classes appear in train, val and test")
print(f"smallest family per set: train {per_family['train'].min()}, "
      f"val {per_family['val'].min()}, test {per_family['test'].min()}")
per_family.head(5)

all 104 classes appear in train, val and test
smallest family per set: train 240, val 30, test 30


,train,val,test,total
family,,,,
lechiket,240,30,30,300
vidro_subdomain,264,33,33,330
sharkbot,280,35,35,350
bigviktor_subdomain,315,40,39,394
sinowal,317,40,39,396


**The DGA/legitimate balance is the same in every set.**

In [7]:
pd.DataFrame({name: part["label"].value_counts(normalize=True).round(4)
              for name, part in splits.items()})

,train,val,test
label,,,
dga,0.5,0.5,0.5
legit,0.5,0.5,0.5


**No domain is in two sets** (so nothing leaks from training into evaluation), and together the sets cover the whole dataset.

In [8]:
train_d, val_d, test_d = (set(p["domain"]) for p in splits.values())
assert not (train_d & val_d) and not (train_d & test_d) and not (val_d & test_d)
assert len(train_d) + len(val_d) + len(test_d) == len(df)
print("no overlap between sets; all", f"{len(df):,}", "domains used once")

no overlap between sets; all 1,999,335 domains used once


## 4. Save the splits

One file per set in `data/splits/`, sorted by family and domain so the files are identical on every machine. `manifest.json` stores each file's row count and SHA-256 fingerprint: if your files have the same fingerprints, you have exactly the same data.

In [9]:
SPLITS.mkdir(parents=True, exist_ok=True)


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


manifest = {"source": BALANCED.name, "seed": SEED, "stratified_by": "family",
            "files": {}}
for name, part in splits.items():
    path = SPLITS / f"{name}.csv.gz"
    part = part.sort_values(["family", "domain"])
    # mtime=0 keeps the gzip header fixed, so the same data gives the same file
    part.to_csv(path, index=False, compression={"method": "gzip", "mtime": 0})
    manifest["files"][path.name] = {
        "rows": len(part),
        "legit": int((part["label"] == "legit").sum()),
        "dga": int((part["label"] == "dga").sum()),
        "sha256": sha256(path),
    }

(SPLITS / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))

{
  "source": "domains_balanced.csv.gz",
  "seed": 42,
  "stratified_by": "family",
  "files": {
    "train.csv.gz": {
      "rows": 1599468,
      "legit": 799731,
      "dga": 799737,
      "sha256": "c920688bf42f5c61e5bcb8275697876271b4550999debf2f8c88ca444b1caf2e"
    },
    "val.csv.gz": {
      "rows": 199933,
      "legit": 99966,
      "dga": 99967,
      "sha256": "92afe1fba3a5dca159b1b3d7e6818160b41f4026c6d2160850a754d570c9ae14"
    },
    "test.csv.gz": {
      "rows": 199934,
      "legit": 99967,
      "dga": 99967,
      "sha256": "303dca886adb4371e3ea7e84d87daacfea09da6500b06cec13d32182f49ff32e"
    }
  }
}


## 5. Reload and verify

This is how every model should load the data: read the three files and check them against the manifest.

In [10]:
manifest = json.loads((SPLITS / "manifest.json").read_text())
loaded = {}
for fname, info in manifest["files"].items():
    path = SPLITS / fname
    assert sha256(path) == info["sha256"], f"{fname} differs from the manifest"
    loaded[fname.split(".")[0]] = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert len(loaded[fname.split(".")[0]]) == info["rows"]
print("all files match the manifest:",
      {name: f"{len(part):,}" for name, part in loaded.items()})

all files match the manifest: {'train': '1,599,468', 'val': '199,933', 'test': '199,934'}
